In [ ]:
#import packages
from pprint import pprint
import tensorflow as tf
import keras
import numpy as np
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.utility as utl
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.windowing as window
import aneurysm_3Dsegmentation_in_CTA.inputPipeline.train_test_split as tts
import aneurysm_3Dsegmentation_in_CTA.model_utility.configuration as conf
import aneurysm_3Dsegmentation_in_CTA.model_utility.metrics as metrics
import segmentation_models_3D as sm3

In [8]:
# loading data tensors
dataSource = "data/CTA nii" 

smallAneurysm , mediumAneurysm , largeAneurysm = tts.dataSplitPerSize(dataSource)

trainSet , testSet = tts.dataSplitPerSample(
    [
        smallAneurysm ,
        mediumAneurysm ,
        largeAneurysm
    ] ,
    testRatio=0.2 ,
    seed=42
)

imgTrainSet , labelTrainSet = tts.dataTensorLoading(trainSet)
imgTestSet  , labelTestSet = tts.dataTensorLoading(testSet)


In [ ]:
# check for image pairs
for name in zip(imgTrainSet , labelTrainSet) :
    print(name[0])
    print(name[1])
    print("============")
print("//////////////////////////////////////////////// test ////////////////////////////////////////////////")
# check for image pairs
for name in zip(imgTestSet , labelTestSet) :
    print(name[0])
    print(name[1])
    print("============")

In [9]:
# pipeline configuring

geo      = utl.randomGeo(p=1)
crop     = utl.volume_crop((128 , 128 , 128))
windower = window.randomMultiWindowStackig(
    default = (200 , 620) ,
    wlRange=(170 , 225) ,
    wwRange=(600 , 650) ,
    p_wl=1 ,
    p_ww=1
)

# wrapping
@tf.py_function(Tout=[tf.float64 , tf.float64])
def rimg(imgPath , labelPath) :
    return utl.read_img(imgPath , labelPath)
def read_img(img , label) :
    imglbl = rimg(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label

@tf.py_function(Tout=[tf.float64 , tf.float64])
def rotate(img , label) :
    img , label = geo.rot(
        img , 
        label ,
        imgOrder=1 ,
        lblOrder=0 ,
        imgCval=-1024 ,
        lblCval=0
    )
    img , label = geo.flip(
        img , 
        label
    )
    return img , label
def rot(img , label) :
    imglbl = rotate(img , label)
    img   = imglbl[0]
    label = imglbl[1]
    return img , label



In [10]:
# dataloaders
dataloaderTrain = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))
    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    #.cache("myCacheTrain")
    #.shuffle(buffer_size=100)
    
    .map(
        rot ,
        num_parallel_calls=4
    )
    .map(
        windower.WindowStacking ,
        num_parallel_calls=4
    )

    .batch(batch_size=2)

    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
)

dataloaderValid = (
    tf.data.Dataset.from_tensor_slices((imgTrainSet , labelTrainSet))

    .map(
        read_img , 
        num_parallel_calls=4
    )
    .map(
        crop.cropping ,
        num_parallel_calls=4
    )

    #.cache("myCacheValid")
    .batch(batch_size=2)
    
    .map(
        utl.normalize ,
        num_parallel_calls=4
    )
)

In [ ]:
cnt=0
# vectorize data model testing

for data in dataloaderTrain.take(10) :
    print(data[0].shape)
    print(data[1].shape)

In [25]:
#loss

binaryFocalLoss = sm3.losses.binary_focal_loss
diceLoss        = sm3.losses.dice_loss 
weightedBinaryFocalDiceLoss = conf.WeightedSumOfLosses(binaryFocalLoss , diceLoss , alpha=0.8)

model = sm3.Unet(
    backbone_name="seresnet18" , 
    input_shape=(128 , 128 , 128 , 3) , 
    classes=1 , 
    activation="sigmoid" ,
    encoder_weights="imagenet" ,
    encoder_freeze=True ,
    decoder_block_type="transpose" ,
    encoder_features=conf.encoderF_d4
)
# model unfreezing
model = conf.unfreeze_model(
    model , 
    conf.unfreeze34_border , 
    keras.src.layers.normalization.batch_normalization.BatchNormalization
)

In [33]:
print(model.summary())

Model: "functional_29"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ data (InputLayer)   │ (None, 128, 128,  │          0 │ -                 │
│                     │ 128, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_data             │ (None, 128, 128,  │          9 │ data[0][0]        │
│ (BatchNormalizatio… │ 128, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding3d_144  │ (None, 134, 134,  │          0 │ bn_data[0][0]     │
│ (ZeroPadding3D)     │ 134, 3)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv0 (Conv3D)      │ (None, 64, 64,    │     65,856 │ zero_padding3d_1… │
│                     │ 64, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn0                 │ (None, 64, 64,    │        256 │ conv0[0][0]       │
│ (BatchNormalizatio… │ 64, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu0 (Activation)  │ (None, 64, 64,    │          0 │ bn0[0][0]         │
│                     │ 64, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding3d_145  │ (None, 66, 66,    │          0 │ relu0[0][0]       │
│ (ZeroPadding3D)     │ 66, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pooling0            │ (None, 32, 32,    │          0 │ zero_padding3d_1… │
│ (MaxPooling3D)      │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_bn1    │ (None, 32, 32,    │        256 │ pooling0[0][0]    │
│ (BatchNormalizatio… │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_relu1  │ (None, 32, 32,    │          0 │ stage1_unit1_bn1… │
│ (Activation)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding3d_146  │ (None, 34, 34,    │          0 │ stage1_unit1_rel… │
│ (ZeroPadding3D)     │ 34, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_conv1  │ (None, 32, 32,    │    110,592 │ zero_padding3d_1… │
│ (Conv3D)            │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_bn2    │ (None, 32, 32,    │        256 │ stage1_unit1_con… │
│ (BatchNormalizatio… │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_relu2  │ (None, 32, 32,    │          0 │ stage1_unit1_bn2… │
│ (Activation)        │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding3d_147  │ (None, 34, 34,    │          0 │ stage1_unit1_rel… │
│ (ZeroPadding3D)     │ 34, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stage1_unit1_conv2  │ (None, 32, 32,    │    110,592 │ zero_padding3d_1… │
│ (Conv3D)            │ 32, 64)           │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 64)        │          0 │ stage1_unit1_con

 Total params: 49,216,370 (187.75 MB)

 Trainable params: 15,918,964 (60.73 MB)

 Non-trainable params: 33,297,406 (127.02 MB)

None


In [32]:
# compilation
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3) ,
    loss = weightedBinaryFocalDiceLoss ,
    metrics=[
        metrics.V_Recall ,
        metrics.dice ,
        metrics.HD
    ]
)

In [29]:
# model training
history = model.fit(
    x=dataloaderTrain ,
    epochs=200 ,
    validation_data=dataloaderValid
)

Epoch 1/200


TypeError: Exception encountered when calling BatchNormalization.call().

[1mFailed to convert elements of [1, 1, 1, 1, None] to Tensor. Consider casting elements to a supported type. See https://www.tensorflow.org/api_docs/python/tf/dtypes for supported TF dtypes.[0m

Arguments received by BatchNormalization.call():
  • inputs=tf.Tensor(shape=(None, None, None, None, None), dtype=float32)
  • training=True
  • mask=None